In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera

df = pd.read_csv("../data/wine_quality_cleaned.csv")
print(df.shape)

In [ ]:
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
coefficients = pd.DataFrame({
    "Coefficient": model.params,
    "P-value": model.pvalues,
    "CI Lower": model.conf_int()[0],
    "CI Upper": model.conf_int()[1],
})
display(coefficients.round(4))

In [ ]:
print("R-squared          :", round(model.rsquared, 4))
print("Adjusted R-squared :", round(model.rsquared_adj, 4))
print("F-statistic        :", round(model.fvalue, 4))
print("F-test p-value     :", model.f_pvalue)

In [ ]:
residuals = model.resid
fitted = model.fittedvalues

plt.figure(figsize=(7, 5))
sns.scatterplot(x=fitted, y=residuals, alpha=0.4)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted (fitted) quality")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")
plt.savefig("../outputs/charts/ols_residuals_vs_fitted.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 5))
sns.histplot(residuals, kde=True)
plt.xlabel("Residuals")
plt.title("Distribution of Residuals")
plt.savefig("../outputs/charts/ols_residual_histogram.png", dpi=150, bbox_inches="tight")
plt.show()


sm.qqplot(residuals, line="45", fit=True)
plt.title("Normal Q-Q Plot of Residuals")
plt.savefig("../outputs/charts/ols_qq_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:

bp_lm, bp_p, bp_f, bp_fp = het_breuschpagan(model.resid, model.model.exog)
jb_stat, jb_p, skew, kurt = jarque_bera(model.resid)
dw = durbin_watson(model.resid)

print("Breusch-Pagan test (constant variance):  LM = %.2f, p = %.3g" % (bp_lm, bp_p))
print("Jarque-Bera test (normal residuals)   :  JB = %.2f, p = %.3g" % (jb_stat, jb_p))
print("Residual skewness = %.3f, kurtosis = %.3f (normal = 0 and 3)" % (skew, kurt))
print("Durbin-Watson statistic               :  %.3f (about 2 = no autocorrelation)" % dw)

In [ ]:
X_vif = X.drop(columns="const")

vif_results = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
display(vif_results.round(4))

In [ ]:
predictors_2 = [p for p in predictors if p != "density"]
X2 = sm.add_constant(df[predictors_2])
model2 = sm.OLS(y, X2).fit()

vif2 = pd.DataFrame({
    "Variable": predictors_2,
    "VIF": [variance_inflation_factor(X2.drop(columns="const").values, i) for i in range(len(predictors_2))],
})
display(vif2.round(4))

comparison = pd.DataFrame({
    "Model 1 coef (with density)": model.params,
    "Model 1 p-value": model.pvalues,
    "Model 2 coef (no density)": model2.params,
    "Model 2 p-value": model2.pvalues,
}).reindex(model.params.index)
display(comparison.round(4))

print("Model 1: R2 = %.4f, adjusted R2 = %.4f" % (model.rsquared, model.rsquared_adj))
print("Model 2: R2 = %.4f, adjusted R2 = %.4f" % (model2.rsquared, model2.rsquared_adj))

In [ ]:
coefficients.to_csv("../outputs/ols_regression_coefficients.csv")
vif_results.to_csv("../outputs/ols_vif_results.csv", index=False)
comparison.to_csv("../outputs/ols_model1_vs_model2.csv")

with open("../outputs/ols_model_summary.txt", "w") as file:
    file.write(model.summary().as_text())

print("OLS summary, coefficients, VIF results and model comparison saved.")

report

1. Objective. The model uses six chemical properties to explain the quality score of white wines (n = 3,961).

2. Regression coefficients (Model 1), holding the other variables constant
alcohol = +0.248: each extra 1% of alcohol is associated with about +0.25 quality points (95% CI 0.208 to 0.287).
volatile acidity = -1.775: each +0.1 g/dm3 is associated with about -0.18 quality points (CI -2.010 to -1.540). The sign agrees with the EDA.
pH = +0.727: a less acidic wine (higher pH) is associated with higher quality (CI 0.558 to 0.895).
citric acid = +0.298 (p = 0.0045) and residual sugar = +0.061 (p < 0.001): small positive coefficients. Note that the simple correlation of residual sugar with quality is *negative* (-0.12); the sign flips once alcohol and density are held constant, which is a sign of multicollinearity.
density = -104.0: the unit is g/cm3, and density only varies by about 0.05 in the data, so a more readable statement is "-0.10 quality points per +0.001 g/cm3". The value is large and unstable because of its high VIF (see below).
The intercept (104.3) is the prediction when density = 0, which is physically meaningless; it only anchors the equation.

3. Predictor significance. All six p-values are below 0.05 (citric acid is the weakest, p = 0.0045), and none of the 95% confidence intervals contains zero. Statistical significance is not the same as importance: with 3,961 observations even small effects are significant.

4. Model fit. R-squared = 0.2872 and adjusted R-squared = 0.2861. The model explains about 29% of the variation in quality; 71% is explained by other factors (taste, grape, production process) or by non-linear relationships. The adjusted value is almost the same, so the six predictors are not just adding noise.

5. Overall significance. The F-statistic is 265.56 with p = 3.1e-286, far below 0.05, so the model as a whole explains quality better than a model with only an intercept.

6. Assumption checks
Linearity / constant variance: the residuals-vs-fitted plot shows parallel diagonal bands because quality is an integer score. The Breusch-Pagan test rejects constant variance (LM = 60.35, p = 3.8e-11), so there is heteroscedasticity; standard errors and p-values are somewhat unreliable.
Normality of residuals: the histogram and Q-Q plot show heavier tails than a normal distribution (kurtosis about 4.08, skewness about -0.07); the Jarque-Bera test rejects normality (JB = 196.4, p = 2.2e-43). With n close to 4,000 the coefficient estimates remain reasonable, but the p-values should be read with care.
Independence: Durbin-Watson = 1.77, close to 2. (The rows are not time ordered, so this check is of limited meaning.)
Multicollinearity: VIFs are 4.23 (alcohol), 11.23 (density), 1.19 (pH), 1.07 (volatile acidity), 1.15 (citric acid) and 5.53 (residual sugar). Density and sugar are problematic.

7. Model 2 (density removed).All VIFs fall below 1.25. R-squared drops only from 0.2872 to 0.2770 (adjusted 0.2761), so density added just 1 percentage point of explanatory power while making the coefficients unstable. The alcohol coefficient rises to 0.377, volatile acidity stays at about -1.90, and citric acid becomes non-significant (p = 0.33). Model 2 is the more trustworthy model for interpreting the individual effects.

8. Limitations
 It shows association, not causation.
 Quality is an ordinal integer score, but OLS treats it as continuous (a ordinal logistic model would be more appropriate).
 Heteroscedasticity and heavy-tailed residuals affect the p-values (robust standard errors, e.g. HC3, would be a good improvement).
 Only 29% of the variance is explained; the linear form may miss non-linear effects and interactions.
 Outliers were kept, and duplicates were removed (notebook 02), which affects the estimates.